# 🎬 Movie Recommendation System — Exploratory Data Analysis
This notebook explores the dataset and visualizes patterns before model training.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.data_loader import load_all

movies, ratings, users = load_all()
print(movies.shape, ratings.shape, users.shape)

## 1. Ratings Distribution

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(ratings['rating'], bins=10, kde=True, color='steelblue')
plt.title('Distribution of Ratings')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 2. Top Rated Movies

In [ ]:
top_movies = (
    ratings.groupby('movie_id')['rating']
    .mean()
    .reset_index()
    .merge(movies[['movie_id','title']], on='movie_id')
    .sort_values('rating', ascending=False)
    .head(10)
)
plt.figure(figsize=(10,5))
sns.barplot(data=top_movies, x='rating', y='title', palette='viridis')
plt.title('Top 10 Highest Rated Movies')
plt.tight_layout()
plt.show()

## 3. Genre Frequency

In [ ]:
genres_exploded = movies['genres'].str.split('|').explode()
genre_counts = genres_exploded.value_counts()

plt.figure(figsize=(10,5))
genre_counts.plot(kind='bar', color='coral')
plt.title('Genre Frequency in Dataset')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. User Activity (Ratings per User)

In [ ]:
user_activity = ratings.groupby('user_id')['movie_id'].count().reset_index()
user_activity.columns = ['user_id', 'num_ratings']

plt.figure(figsize=(8,4))
sns.barplot(data=user_activity, x='user_id', y='num_ratings', palette='mako')
plt.title('Number of Ratings per User')
plt.xlabel('User ID')
plt.ylabel('Ratings Count')
plt.tight_layout()
plt.show()

## 5. User-Movie Sparsity Heatmap

In [ ]:
matrix = ratings.pivot_table(index='user_id', columns='movie_id', values='rating').fillna(0)
plt.figure(figsize=(14,5))
sns.heatmap(matrix, cmap='YlOrRd', linewidths=0.5, linecolor='gray')
plt.title('User-Movie Rating Matrix (0 = not rated)')
plt.tight_layout()
plt.show()
sparsity = 1 - (ratings.shape[0] / (matrix.shape[0] * matrix.shape[1]))
print(f'Matrix sparsity: {sparsity:.2%}')